# RouteRL Quickstart

We simulate a simple network topology where humans and later AVs make routing decisions to maximize their rewards (i.e., minimize travel times) over a sequence of days.

* For the first 100 days, we model a human-driven system, where drivers update their routing policies using behavioral models to optimize rewards.
* Each day, we simulate the impact of joint actions using the [`SUMO`](https://eclipse.dev/sumo/) traffic simulator, which returns the reward for each agent.
* After 100 days, we introduce 10 `Autononmous Vehicles` (AVs) as `Petting Zoo` agents, allowing them to use any `MARL` algorithm to maximise rewards. In this tutorial, we use a trained policy from the Independent Deep Q-Learning (IDQN) algorithm.
* Finally, we analyse basic results from the simulation.
  

# Tutorial Outline

* Establishing the Connection with SUMO

* Initializing the Traffic Environment
  - Define the `TrafficEnvironment`, which initializes human agents and generates the routes agents will travel within the network.

* Training Human Agents
  - Train human-driven vehicles to navigate the environment efficiently using human behavioural models from transportation research.

* Introducing Autonomous Vehicles (AVs)
  - Transform a subset of human agents into AVs.
  - AVs select their routes using a pre-trained policy based on the IDQN algorithm.

* Analyzing the Impact of AVs
  - Evaluate the effects of AV introduction on human travel time, congestion, and CO₂ emissions.
  - Demonstrate how AV deployment can potentially increase travel delays and environmental impact.

<p align="center">
  <img src="../../docs/img/two_route_net_1.png" alt="Two-route network" />
  <img src="../../docs/img/two_route_net_1_2.png" alt="Two-route network" />
</p>  

#### Import libraries

In [1]:
import sys
import os
import pandas as pd
import torch
from tensordict.nn import TensorDictModule, TensorDictSequential
from torchrl.envs.libs.pettingzoo import PettingZooWrapper
from torchrl.envs.transforms import TransformedEnv, RewardSum
from torchrl.envs.utils import check_env_specs
from torch import nn
from torchrl._utils import logger as torchrl_logger
from torchrl.collectors import SyncDataCollector
from torchrl.data import TensorDictReplayBuffer
from torchrl.data.replay_buffers.samplers import SamplerWithoutReplacement
from torchrl.data.replay_buffers.storages import LazyTensorStorage
from torchrl.modules import EGreedyModule, QValueModule, SafeSequential
from torchrl.modules.models.multiagent import MultiAgentMLP
from torchrl.objectives import SoftUpdate, ValueEstimators, DQNLoss

from routerl import TrafficEnvironment

os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"

#### Define hyperparameters

> Users can customize parameters for the `TrafficEnvironment` class by consulting the [`routerl/environment/params.json`](https://github.com/COeXISTENCE-PROJECT/RouteRL/blob/4f4bc0a90d821e95b7193b00c93d6aaf10b34f41/routerl/environment/params.json) file. Based on its contents, they can create a dictionary with their preferred settings and pass it as an argument to the `TrafficEnvironment` class.

> **In this repository we don't recommend adjusting the number of agents, because the policy is trained for 10 AV agents.**


In [2]:
device = (
    torch.device(0)
    if torch.cuda.is_available()
    else torch.device("cpu")
)
print("device is: ", device)

# Sampling
frames_per_batch = 100  # Number of team frames collected per training iteration
n_iters = 10 # Number of sampling and training iterations - the episodes the plotter plots
total_frames = frames_per_batch * n_iters

# Training
num_epochs = 1  # Number of optimization steps per training iteration 
minibatch_size = 16  # Size of the mini-batches in each optimization step
lr = 3e-3 # Learning rate
max_grad_norm = 5.0  # Maximum norm for the gradients
memory_size = 5_000  # Size of the replay buffer
tau =  0.05
gamma = 0.99  # discount factor
exploration_fraction = 1/3 # Fraction of frames over which the exploration rate is annealed

eps = 1 - tau
eps_init = 0.99
eps_end = 0

mlp_depth = 2
mlp_cells = 32

# Human learning phase
human_learning_episodes = 100

env_params = {
    "agent_parameters" : {
        "num_agents" : 100,
        "new_machines_after_mutation": 10, # the number of human agents that will mutate to AVs
        "human_parameters" : {
            "model" : "gawron"
        },
        "machine_parameters" :
        {
            "behavior" : "selfish",
        }
    },
    "simulator_parameters" : {
        "network_name" : "two_route_trafficlight",
        "simulation_timesteps" : 180,
        "sumo_type" : "sumo",
    },  
    "plotter_parameters" : {
        "phases" : [0, human_learning_episodes], # the number of episodes human learning will take
    },
}

device is:  cpu


#### Environment initialization

In our setup, road networks initially consist of human agents, with AVs introduced later.

- The `TrafficEnvironment` environment is firstly initialized.
- The traffic network is instantiated and the paths between designated origin and destination points are determined.
- The drivers/agents objects are created.

In [3]:
env = TrafficEnvironment(seed=42, **env_params)

[CONFIRMED] Environment variable exists: SUMO_HOME
[SUCCESS] Added module directory: C:\Program Files (x86)\Eclipse\Sumo\tools


> Available paths create using the [Janux](https://github.com/COeXISTENCE-PROJECT/JanuX) framework.

<p >
  <img src="plots_saved/0_0.png" width="600" />
</p>  

In [4]:
print("Number of total agents is: ", len(env.all_agents), "\n")
print("Number of human agents is: ", len(env.human_agents), "\n")
print("Number of machine agents (autonomous vehicles) is: ", len(env.machine_agents), "\n")

Number of total agents is:  100 

Number of human agents is:  100 

Number of machine agents (autonomous vehicles) is:  0 



> Reset the environment and the connection with SUMO

In [5]:
env.start()

FatalTraCIError: Connection closed by SUMO.

#### Human learning

In [ ]:
for episode in range(human_learning_episodes):
    env.step() # all the human agents execute an action in the environment

> Average travel time of human agents during their training process.

<p align="center">
  <img src="plots_saved/human_learning.png"/>
</p> 

> Show the initial `.csv` file saved that contain the information about the agents available in the system.


In [ ]:
df = pd.read_csv("training_records/episodes/ep1.csv")
df


,travel_time,id,kind,action,origin,destination,start_time,reward,cost_table
0,5.183333,0,Human,1,0,0,99,-5.183333,"0.2217422606191504,-2.364644828413727"
1,1.200000,1,Human,1,0,0,58,-1.200000,"0.2217422606191504,-0.3729781617470602"
2,7.550000,2,Human,1,0,0,112,-7.550000,"0.2217422606191504,-3.5479781617470603"
3,7.800000,3,Human,1,0,0,118,-7.800000,"0.2217422606191504,-3.6729781617470603"
4,1.183333,4,Human,1,0,0,31,-1.183333,"0.2217422606191504,-0.3646448284137269"
...,...,...,...,...,...,...,...,...,...
95,1.116667,95,Human,1,0,0,46,-1.116667,"0.2217422606191504,-0.3313114950803936"
96,1.116667,96,Human,1,0,0,50,-1.116667,"0.2217422606191504,-0.3313114950803936"
97,1.216667,97,Human,1,0,0,60,-1.216667,"0.2217422606191504,-0.3813114950803935"
98,6.050000,98,Human,1,0,0,101,-6.050000,"0.2217422606191504,-2.7979781617470603"


#### Mutation

> Mutation: a portion of human agents are converted into machine agents (autonomous vehicles). 

In [ ]:
env.mutation()

In [ ]:
print("Number of total agents is: ", len(env.all_agents), "\n")
print("Number of human agents is: ", len(env.human_agents), "\n")
print("Number of machine agents (autonomous vehicles) is: ", len(env.machine_agents), "\n")

Number of total agents is:  100 

Number of human agents is:  90 

Number of machine agents (autonomous vehicles) is:  10 



In [ ]:
env.machine_agents

[Machine 1,
 Machine 15,
 Machine 10,
 Machine 91,
 Machine 22,
 Machine 73,
 Machine 5,
 Machine 52,
 Machine 81,
 Machine 77]

> In order to employ the `TorchRL` library in our environment we need to use their `PettingZooWrapper` function.

In [ ]:
group = {'agents': [str(machine.id) for machine in env.machine_agents]}

env = PettingZooWrapper(
    env=env,
    use_mask=True,
    categorical_actions=True,
    done_on_any = False,
    group_map=group,
    device=device
)

> Define the neural network used by `TorchRL`.

> We saved only the learnable parameters of the network, so we need to redefine its architecture before loading them.

In [ ]:
net = MultiAgentMLP(
        n_agent_inputs=env.observation_spec["agents", "observation"].shape[-1],
        n_agent_outputs=env.action_spec.space.n,
        n_agents=env.n_agents,
        centralised=False,
        share_params=False,
        depth=mlp_depth,
        num_cells=mlp_cells,
        activation_class=nn.ReLU,
    )

module = TensorDictModule(
        net, in_keys=[("agents", "observation")], out_keys=[("agents", "action_value")]
)

value_module = QValueModule(
    action_value_key=("agents", "action_value"),
    out_keys=[
        env.action_key,
        ("agents", "action_value"),
        ("agents", "chosen_action_value"),
    ],
    spec=env.action_spec,
    action_space=None,
)

qnet = SafeSequential(module, value_module)

qnet_explore = TensorDictSequential(
    qnet,
    EGreedyModule(
        eps_init=eps_init,
        eps_end=eps_end,
        annealing_num_steps=int(total_frames * exploration_fraction),
        action_key=env.action_key,
        spec=env.action_spec,
    ),
)

> Confirm that the policy defined runs properly.

In [ ]:
qnet_explore.eval()
num_test_episodes = 1

for episode in range(num_test_episodes): # run rollous in the environment using the already trained policy
    env.rollout(len(env.machine_agents), policy=qnet_explore)

> Load the trainable parameters of the policy.

In [ ]:
n_state_dict = torch.load("policy_checkpoint.pt", map_location=torch.device("cpu"))

In [ ]:
qnet_explore.load_state_dict(n_state_dict, strict=False)  


<All keys matched successfully>

> Human and AV agents interact with the environment over multiple episodes, with AVs following a trained policy.

In [ ]:
num_test_episodes = 100

for episode in range(num_test_episodes): # run rollous in the environment using the already trained policy
    env.rollout(len(env.machine_agents), policy=qnet)

> Show the first `.csv` file saved after the mutation that contains the information about the agents available in the system after the mutation.

In [ ]:
df = pd.read_csv("training_records/episodes/ep101.csv")
df

,travel_time,id,kind,action,origin,destination,start_time,reward,cost_table
0,0.650000,1,AV,0,0,0,58,-0.650000,"0,0"
1,2.133333,15,AV,1,0,0,64,-2.133333,"0,0"
2,4.916667,10,AV,1,0,0,116,-4.916667,"0,0"
3,2.416667,91,AV,1,0,0,87,-2.416667,"0,0"
4,5.150000,22,AV,0,0,0,126,-5.150000,"0,0"
...,...,...,...,...,...,...,...,...,...
95,0.516667,95,Human,0,0,0,46,-0.516667,"-0.55,-0.6906557475401969"
96,0.483333,96,Human,0,0,0,50,-0.483333,"-0.5333333333333332,-0.6989890808735302"
97,1.066667,97,Human,1,0,0,60,-1.066667,"-1.2472822174226061,-1.0833333333333335"
98,3.800000,98,Human,0,0,0,101,-3.800000,"-3.783333333333333,-3.8573224142068634"


#### Plot results 

>This will be shown in the `\plots` folder.

In [ ]:
env.plot_results()

> The results highlight a critical challenge in AV deployment: rather than improving traffic flow, AVs may increase travel time for human drivers. This suggests potential inefficiencies in mixed traffic conditions due to differences in driving behavior. Understanding these effects is essential for designing better reinforcement learning strategies, informing policymakers, and optimizing AV integration to prevent increased congestion and CO₂ emissions.


| |  |
|---------|---------|
| **Action shifts of human and AV agents** ![](plots_saved/actions_shifts.png) | **Action shifts of all vehicles in the network** ![](plots_saved/actions.png) |
| ![](plots_saved/rewards.png) | ![](plots_saved/travel_times.png) |


<p align="center">
  <img src="plots_saved/tt_dist.png" width="700" />
</p>


> Interrupt the connection with `SUMO`.

In [ ]:
env.stop_simulation()